# 01. Explorar variáveis — Splink

Profile Splink, blocking rules e draft de settings.

**EDA descritiva (gráficos, missing, top nomes):** use [`01_analise_descritiva.ipynb`](01_analise_descritiva.ipynb).


In [ ]:
import sys, json
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from config import (
    SPLINK_SETTINGS_DRAFT,
    USE_PHONETIC_STRIP_VOWELS,
    get_connection,
    get_splink_db_api,
    print_paths,
    require_tables,
)

print_paths()
con = get_connection()
require_tables(con, ['registro_unificado', 'ground_truth_clusters'], notebook_origem='00')


In [ ]:
gt_cov = con.execute('''
WITH pairs AS (
    SELECT g1.unique_id AS id_censo, g2.unique_id AS id_cpf
    FROM ground_truth_clusters g1
    JOIN ground_truth_clusters g2 ON g1.cluster = g2.cluster AND g1.cluster LIKE 'gt_%'
    WHERE g1.unique_id LIKE 'censo_%' AND g2.unique_id LIKE 'cpf_%'
),
in_stack AS (
    SELECT p.*,
        EXISTS (SELECT 1 FROM registro_unificado r WHERE r.unique_id = p.id_censo) AS censo_ok,
        EXISTS (SELECT 1 FROM registro_unificado r WHERE r.unique_id = p.id_cpf) AS cpf_ok
    FROM pairs p
)
SELECT COUNT(*) AS n_pares_coorte,
    SUM(CASE WHEN censo_ok AND cpf_ok THEN 1 ELSE 0 END) AS n_pares_no_subset,
    ROUND(100.0 * SUM(CASE WHEN censo_ok AND cpf_ok THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_cobertura
FROM in_stack
''').df()
gt_cov


## Profile e análise de blocking

Gráfico cumulativo de comparações + maiores blocos por regra (sem substr).

In [ ]:
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
from splink.exploratory import profile_columns, n_largest_blocks

from config import SPLINK_INPUT_VIEW, materialize_splink_input

# Blocking rules — colunas completas (sem substr)
blocking_rules = [
    block_on('primeiro_nome', 'ultimo_nome'),
    block_on('ultimo_nome', 'data_nascimento'),
    block_on('primeiro_nome', 'data_nascimento'),
    block_on('cep', 'ultimo_nome'),
    block_on('cep', 'primeiro_nome'),
    block_on('nome_mae', 'data_nascimento'),
]

materialize_splink_input(con)
db_api = get_splink_db_api(con)

SPLINK_ANALYSIS_SAMPLE_N = 100_000
n_total = con.execute('SELECT COUNT(*) FROM registro_unificado').fetchone()[0]
use_sample = n_total > SPLINK_ANALYSIS_SAMPLE_N
analysis_table = SPLINK_INPUT_VIEW

if use_sample:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE splink_analysis_sample AS
    SELECT * FROM {SPLINK_INPUT_VIEW}
    USING SAMPLE {SPLINK_ANALYSIS_SAMPLE_N} ROWS
    ''')
    analysis_table = 'splink_analysis_sample'
    print(f'Amostra blocking/profile: {SPLINK_ANALYSIS_SAMPLE_N:,} de {n_total:,}')

profile_columns(
    con.execute(f'SELECT * FROM {analysis_table}').df(),
    db_api,
    column_expressions=['primeiro_nome', 'ultimo_nome', 'nome_completo_phon', 'cep', 'data_nascimento', 'idade'],
)

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=analysis_table,
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type='dedupe_only',
)

for rule in blocking_rules:
    print(rule)
    n_largest_blocks(table_or_tables=analysis_table, blocking_rule=rule, db_api=db_api, n=5)


In [ ]:
draft = {
    'link_type': 'dedupe_only',
    'cpf_in_comparisons': False,
    'phonetic_basic_columns': ['nome_completo_phon', 'primeiro_nome_phon', 'ultimo_nome_phon'],
    'phonetic_strip_vowels': USE_PHONETIC_STRIP_VOWELS,
    'blocking_rules': [
        'primeiro_nome + ultimo_nome',
        'ultimo_nome + data_nascimento',
        'primeiro_nome + data_nascimento',
        'cep + ultimo_nome',
        'cep + primeiro_nome',
        'nome_mae + data_nascimento',
    ],
}
SPLINK_SETTINGS_DRAFT.parent.mkdir(parents=True, exist_ok=True)
SPLINK_SETTINGS_DRAFT.write_text(json.dumps(draft, indent=2, ensure_ascii=False))
print('Salvo:', SPLINK_SETTINGS_DRAFT)
con.close()
